# CReST: Design-Robustness Refinement

Develops training-only residualisation and system-equal assessment-design calibration and verifies retention of disjoint-item creative signal.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# CReST design-robust refinement environment

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)


# Paths

BASE_DIR = Path.cwd()

HANDOFF_DIR = (
    BASE_DIR
    / "processed"
    / "crest_v5_handoff"
)

RESULTS_DIR = (
    BASE_DIR
    / "results"
)

FIG_DIR = (
    BASE_DIR
    / "figures"
)


RESULTS_DIR.mkdir(
    exist_ok=True
)

FIG_DIR.mkdir(
    exist_ok=True
)


# Configuration

with open(
    HANDOFF_DIR / "handoff_config.json",
    "r",
    encoding="utf-8"
) as f:
    CONFIG = json.load(f)


GLOBAL_SEED = int(
    CONFIG["global_seed"]
)


random.seed(
    GLOBAL_SEED
)

np.random.seed(
    GLOBAL_SEED
)


print(
    "Working directory:",
    BASE_DIR
)

print(
    "Handoff directory:",
    HANDOFF_DIR
)

print(
    "Environment initialised."
)

Working directory: PROJECT_ROOT
Handoff directory: PROJECT_ROOT\processed\crest_v5_handoff
CReST refinement notebook ready.


In [ ]:
# Load only data needed for design-leakage refinement

credit_tokens = np.load(
    HANDOFF_DIR / "credit_tokens.npy",
    mmap_mode="r"
)

behaviour_tokens = np.load(
    HANDOFF_DIR / "behaviour_tokens.npy",
    mmap_mode="r"
)

performance_mask = np.load(
    HANDOFF_DIR / "performance_mask.npy",
    mmap_mode="r"
)

interaction_mask = np.load(
    HANDOFF_DIR / "interaction_mask.npy",
    mmap_mode="r"
)

process_tensor = np.load(
    HANDOFF_DIR / "process_tensor.npy",
    mmap_mode="r"
)

booklet_nuisance_ids = np.load(
    HANDOFF_DIR / "booklet_nuisance_ids.npy",
    mmap_mode="r"
)

item_domain_ids = np.load(
    HANDOFF_DIR / "item_domain_ids.npy",
    mmap_mode="r"
)

item_ideation_ids = np.load(
    HANDOFF_DIR / "item_ideation_ids.npy",
    mmap_mode="r"
)

primary_meta = pd.read_pickle(
    HANDOFF_DIR / "primary_meta.pkl"
)


# Fold 0

fold_values = (
    primary_meta[
        "country_fold"
    ].to_numpy()
)

train_idx = np.where(
    ~np.isin(
        fold_values,
        [0, 1]
    )
)[0]

validation_idx = np.where(
    fold_values == 1
)[0]

test_idx = np.where(
    fold_values == 0
)[0]


# Recover the 10 design-signature mapping

BOOKLET_TO_SIGNATURE = np.array(
    [
        4, 5, 9, 3, 1,
        7, 2, 8, 0, 6,

        4, 5, 9, 3, 1,
        7, 2, 8, 0, 6,

        4, 5, 9, 3, 1,
        7, 2, 8, 0, 6
    ],
    dtype=np.int64
)


design_signature_ids = (
    BOOKLET_TO_SIGNATURE[
        booklet_nuisance_ids
    ]
)


print(
    "Train:",
    len(train_idx)
)

print(
    "Validation:",
    len(validation_idx)
)

print(
    "Test untouched:",
    len(test_idx)
)

print(
    "Design signatures:",
    len(
        np.unique(
            design_signature_ids
        )
    )
)

Train: 99966
Validation: 20949
Test untouched: 21649
Design signatures: 10


In [ ]:
# Train-only item-conditioned residualization

N_ITEMS = credit_tokens.shape[1]
N_PROCESSES = 3
N_PROCESS_CHANNELS = 3

EPS = 1e-6


# Training-only baseline containers

credit_item_mean = np.zeros(
    N_ITEMS,
    dtype=np.float32
)

behaviour_item_rates = np.zeros(
    (
        N_ITEMS,
        3
    ),
    dtype=np.float32
)

process_item_mean = np.zeros(
    (
        N_ITEMS,
        N_PROCESS_CHANNELS
    ),
    dtype=np.float32
)

process_item_std = np.ones(
    (
        N_ITEMS,
        N_PROCESS_CHANNELS
    ),
    dtype=np.float32
)


# Learn item baselines from TRAINING systems only

for item in range(
    N_ITEMS
):

    # Credit
    # 1=no, 2=partial, 3=full
    # -> 0, .5, 1

    mask = np.asarray(
        performance_mask[
            train_idx,
            item
        ],
        dtype=bool
    )

    tokens = np.asarray(
        credit_tokens[
            train_idx,
            item
        ]
    )


    score = (
        tokens.astype(
            np.float32
        )
        - 1.0
    ) / 2.0


    if mask.any():

        credit_item_mean[item] = (
            score[
                mask
            ].mean()
        )


    # Behaviour
    # 1=scorable
    # 2=no response
    # 3=not reached

    b = np.asarray(
        behaviour_tokens[
            train_idx,
            item
        ]
    )

    bmask = np.asarray(
        interaction_mask[
            train_idx,
            item
        ],
        dtype=bool
    )


    if bmask.any():

        for k, token in enumerate(
            [1, 2, 3]
        ):

            behaviour_item_rates[
                item,
                k
            ] = (
                b[
                    bmask
                ]
                == token
            ).mean()


    # Process
    #
    # process_tensor contains log1p-transformed values.
    # NaN = unavailable.

    for channel in range(
        N_PROCESS_CHANNELS
    ):

        x = np.asarray(
            process_tensor[
                train_idx,
                item,
                channel
            ],
            dtype=np.float32
        )


        observed = np.isfinite(
            x
        )


        if observed.any():

            values = x[
                observed
            ]


            mean = float(
                values.mean()
            )

            std = float(
                values.std(
                    ddof=0
                )
            )


            if (
                not np.isfinite(std)
                or
                std < EPS
            ):
                std = 1.0


            process_item_mean[
                item,
                channel
            ] = mean

            process_item_std[
                item,
                channel
            ] = std


print(
    "Train-only item baselines learned."
)

print(
    "Credit means range:",
    (
        float(
            credit_item_mean.min()
        ),
        float(
            credit_item_mean.max()
        )
    )
)

print(
    "Process SD minimum:",
    float(
        process_item_std.min()
    )
)

Train-only item baselines learned.
Credit means range: (0.2417212724685669, 0.6911605000495911)
Process SD minimum: 0.5045149326324463


In [ ]:
# Build residualized GDI/GCI/EII summaries
# No neural-network training.

def build_residual_summary_features(
    indices
):

    n = len(indices)

    credit_residual_features = []
    behaviour_residual_features = []
    process_residual_features = []


    # Residualise item-level signals

    for process_id in range(
        N_PROCESSES
    ):

        item_sel = (
            np.asarray(
                item_ideation_ids
            )
            == process_id
        )

        selected_items = np.where(
            item_sel
        )[0]


        # CREDIT residuals

        per_item_credit = []
        per_item_credit_mask = []


        for item in selected_items:

            tokens = np.asarray(
                credit_tokens[
                    indices,
                    item
                ]
            )


            mask = np.asarray(
                performance_mask[
                    indices,
                    item
                ],
                dtype=bool
            )


            score = (
                tokens.astype(
                    np.float32
                )
                - 1.0
            ) / 2.0


            residual = (
                score
                -
                credit_item_mean[
                    item
                ]
            )


            residual[
                ~mask
            ] = 0.0


            per_item_credit.append(
                residual
            )

            per_item_credit_mask.append(
                mask
            )


        credit_matrix = np.column_stack(
            per_item_credit
        )

        credit_mask_matrix = np.column_stack(
            per_item_credit_mask
        )


        credit_residual_mean = np.divide(

            (
                credit_matrix
                * credit_mask_matrix
            ).sum(
                axis=1
            ),

            credit_mask_matrix.sum(
                axis=1
            ),

            out=np.zeros(
                n,
                dtype=np.float32
            ),

            where=(
                credit_mask_matrix.sum(
                    axis=1
                )
                > 0
            )
        )


        credit_residual_features.append(
            credit_residual_mean
        )


        # BEHAVIOUR residuals

        for behaviour_k, token in enumerate(
            [1, 2, 3]
        ):

            per_item_behaviour = []
            per_item_behaviour_mask = []


            for item in selected_items:

                b = np.asarray(
                    behaviour_tokens[
                        indices,
                        item
                    ]
                )


                mask = np.asarray(
                    interaction_mask[
                        indices,
                        item
                    ],
                    dtype=bool
                )


                indicator = (
                    b == token
                ).astype(
                    np.float32
                )


                residual = (
                    indicator
                    -
                    behaviour_item_rates[
                        item,
                        behaviour_k
                    ]
                )


                residual[
                    ~mask
                ] = 0.0


                per_item_behaviour.append(
                    residual
                )

                per_item_behaviour_mask.append(
                    mask
                )


            behaviour_matrix = (
                np.column_stack(
                    per_item_behaviour
                )
            )


            behaviour_mask_matrix = (
                np.column_stack(
                    per_item_behaviour_mask
                )
            )


            behaviour_residual_mean = np.divide(

                (
                    behaviour_matrix
                    * behaviour_mask_matrix
                ).sum(
                    axis=1
                ),

                behaviour_mask_matrix.sum(
                    axis=1
                ),

                out=np.zeros(
                    n,
                    dtype=np.float32
                ),

                where=(
                    behaviour_mask_matrix.sum(
                        axis=1
                    )
                    > 0
                )
            )


            behaviour_residual_features.append(
                behaviour_residual_mean
            )


        # PROCESS residuals

        for channel in range(
            N_PROCESS_CHANNELS
        ):

            per_item_process = []
            per_item_process_mask = []


            for item in selected_items:

                x = np.asarray(
                    process_tensor[
                        indices,
                        item,
                        channel
                    ],
                    dtype=np.float32
                )


                mask = np.isfinite(
                    x
                )


                residual = np.zeros(
                    n,
                    dtype=np.float32
                )


                residual[
                    mask
                ] = (

                    x[
                        mask
                    ]

                    -

                    process_item_mean[
                        item,
                        channel
                    ]

                ) / (

                    process_item_std[
                        item,
                        channel
                    ]

                    + EPS
                )


                per_item_process.append(
                    residual
                )

                per_item_process_mask.append(
                    mask
                )


            process_matrix = np.column_stack(
                per_item_process
            )


            process_mask_matrix = np.column_stack(
                per_item_process_mask
            )


            process_residual_mean = np.divide(

                (
                    process_matrix
                    * process_mask_matrix
                ).sum(
                    axis=1
                ),

                process_mask_matrix.sum(
                    axis=1
                ),

                out=np.zeros(
                    n,
                    dtype=np.float32
                ),

                where=(
                    process_mask_matrix.sum(
                        axis=1
                    )
                    > 0
                )
            )


            process_residual_features.append(
                process_residual_mean
            )


    return {

        "credit_residual":
            np.column_stack(
                credit_residual_features
            ).astype(
                np.float32
            ),

        "behaviour_residual":
            np.column_stack(
                behaviour_residual_features
            ).astype(
                np.float32
            ),

        "process_residual":
            np.column_stack(
                process_residual_features
            ).astype(
                np.float32
            )
    }


train_residual = (
    build_residual_summary_features(
        train_idx
    )
)

validation_residual = (
    build_residual_summary_features(
        validation_idx
    )
)


print(
    "Train residual shapes:"
)

for name, values in (
    train_residual.items()
):

    print(
        f"{name:24s}",
        values.shape
    )

Train residual shapes:
credit_residual          (99966, 3)
behaviour_residual       (99966, 9)
process_residual         (99966, 9)


In [ ]:
# Probe design leakage after train-only item residualization

y_train_signature = (
    design_signature_ids[
        train_idx
    ]
)

y_validation_signature = (
    design_signature_ids[
        validation_idx
    ]
)


def residual_probe(
    X_train,
    X_val,
    name
):

    clf = make_pipeline(

        StandardScaler(),

        LogisticRegression(
            max_iter=500,
            solver="lbfgs",
            class_weight="balanced",
            random_state=GLOBAL_SEED
        )
    )


    clf.fit(
        X_train,
        y_train_signature
    )


    pred = clf.predict(
        X_val
    )


    return {

        "feature_set":
            name,

        "n_features":
            X_train.shape[1],

        "accuracy":
            accuracy_score(
                y_validation_signature,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation_signature,
                pred
            ),

        "macro_f1":
            f1_score(
                y_validation_signature,
                pred,
                average="macro"
            )
    }


residual_probe_results = []


# Separate views

for name in [
    "credit_residual",
    "behaviour_residual",
    "process_residual"
]:

    residual_probe_results.append(

        residual_probe(

            train_residual[name],

            validation_residual[name],

            name
        )
    )


# Combined residual representation

X_train_residual_all = np.column_stack(
    [
        train_residual[
            "credit_residual"
        ],

        train_residual[
            "behaviour_residual"
        ],

        train_residual[
            "process_residual"
        ]
    ]
)


X_validation_residual_all = np.column_stack(
    [
        validation_residual[
            "credit_residual"
        ],

        validation_residual[
            "behaviour_residual"
        ],

        validation_residual[
            "process_residual"
        ]
    ]
)


residual_probe_results.append(

    residual_probe(

        X_train_residual_all,

        X_validation_residual_all,

        "all_residual_views"
    )
)


residual_probe_df = pd.DataFrame(
    residual_probe_results
)


print(
    residual_probe_df.to_string(
        index=False
    )
)


print(
    "\nReferences:"
)

print(
    "Chance BA                         : 0.1000"
)

print(
    "Mask-free raw summaries BA        : 0.3183"
)

print(
    "Current V6 z_general design BA    : 0.3756"
)

       feature_set  n_features  accuracy  balanced_accuracy  macro_f1
   credit_residual           3  0.120388           0.120558  0.085407
behaviour_residual           9  0.141821           0.143077  0.122749
  process_residual           9  0.209509           0.210074  0.190749
all_residual_views          21  0.226646           0.227259  0.212018

References:
Chance BA                         : 0.1000
Mask-free raw summaries BA        : 0.3183
Current V6 z_general design BA    : 0.3756


In [ ]:
# Train-only item-wise quantile normalization for process values
# No neural-network training.

from scipy.stats import norm


# Learn empirical training distributions

process_train_sorted = {}


for item in range(N_ITEMS):

    for channel in range(
        N_PROCESS_CHANNELS
    ):

        x = np.asarray(
            process_tensor[
                train_idx,
                item,
                channel
            ],
            dtype=np.float32
        )


        observed = np.isfinite(
            x
        )


        values = np.sort(
            x[
                observed
            ]
        )


        process_train_sorted[
            (
                item,
                channel
            )
        ] = values


print(
    "Empirical process distributions learned:",
    len(
        process_train_sorted
    )
)

Empirical process distributions learned: 96


In [ ]:
# Transform process values using training-only empirical CDFs

def empirical_normal_score(
    values,
    reference_sorted
):

    result = np.zeros(
        len(values),
        dtype=np.float32
    )


    observed = np.isfinite(
        values
    )


    if not observed.any():

        return result, observed


    ref = reference_sorted

    n_ref = len(
        ref
    )


    if n_ref == 0:

        return result, observed


    x = values[
        observed
    ]


    # Mid-rank empirical percentile
    left = np.searchsorted(
        ref,
        x,
        side="left"
    )

    right = np.searchsorted(
        ref,
        x,
        side="right"
    )


    rank_mid = (
        left
        +
        right
    ) / 2.0


    p = (
        rank_mid
        + 0.5
    ) / (
        n_ref
        + 1.0
    )


    # Prevent infinite normal scores
    p = np.clip(
        p,
        1e-4,
        1.0 - 1e-4
    )


    result[
        observed
    ] = norm.ppf(
        p
    ).astype(
        np.float32
    )


    return (
        result,
        observed
    )


def build_quantile_process_features(
    indices
):

    n = len(
        indices
    )

    features = []


    for process_id in range(
        N_PROCESSES
    ):

        selected_items = np.where(
            np.asarray(
                item_ideation_ids
            )
            == process_id
        )[0]


        for channel in range(
            N_PROCESS_CHANNELS
        ):

            item_scores = []
            item_masks = []


            for item in selected_items:

                x = np.asarray(
                    process_tensor[
                        indices,
                        item,
                        channel
                    ],
                    dtype=np.float32
                )


                scores, observed = (
                    empirical_normal_score(

                        x,

                        process_train_sorted[
                            (
                                item,
                                channel
                            )
                        ]
                    )
                )


                item_scores.append(
                    scores
                )

                item_masks.append(
                    observed
                )


            score_matrix = np.column_stack(
                item_scores
            )

            mask_matrix = np.column_stack(
                item_masks
            )


            denominator = (
                mask_matrix.sum(
                    axis=1
                )
            )


            mean_score = np.divide(

                (
                    score_matrix
                    * mask_matrix
                ).sum(
                    axis=1
                ),

                denominator,

                out=np.zeros(
                    n,
                    dtype=np.float32
                ),

                where=(
                    denominator > 0
                )
            )


            features.append(
                mean_score
            )


    return np.column_stack(
        features
    ).astype(
        np.float32
    )


train_process_quantile = (
    build_quantile_process_features(
        train_idx
    )
)


validation_process_quantile = (
    build_quantile_process_features(
        validation_idx
    )
)


print(
    "Train quantile process:",
    train_process_quantile.shape
)

print(
    "Validation quantile process:",
    validation_process_quantile.shape
)

Train quantile process: (99966, 9)
Validation quantile process: (20949, 9)


In [ ]:
# Design-signature probe after process quantile normalization

quantile_probe_results = []


# Process only

quantile_probe_results.append(

    residual_probe(

        train_process_quantile,

        validation_process_quantile,

        "process_quantile"
    )
)


# Credit residual + behaviour residual + quantile process

X_train_quantile_all = np.column_stack(
    [
        train_residual[
            "credit_residual"
        ],

        train_residual[
            "behaviour_residual"
        ],

        train_process_quantile
    ]
)


X_validation_quantile_all = np.column_stack(
    [
        validation_residual[
            "credit_residual"
        ],

        validation_residual[
            "behaviour_residual"
        ],

        validation_process_quantile
    ]
)


quantile_probe_results.append(

    residual_probe(

        X_train_quantile_all,

        X_validation_quantile_all,

        "credit_resid+behaviour_resid+process_quantile"
    )
)


quantile_probe_df = pd.DataFrame(
    quantile_probe_results
)


print(
    quantile_probe_df.to_string(
        index=False
    )
)


print(
    "\nReferences:"
)

print(
    "Chance BA                       : 0.1000"
)

print(
    "Raw mask-free combined BA       : 0.3183"
)

print(
    "Residualized combined BA        : 0.2273"
)

print(
    "Residualized process BA         : 0.2101"
)

                                  feature_set  n_features  accuracy  balanced_accuracy  macro_f1
                             process_quantile           9  0.210464           0.211049  0.192425
credit_resid+behaviour_resid+process_quantile          21  0.226455           0.227072  0.212470

References:
Chance BA                       : 0.1000
Raw mask-free combined BA       : 0.3183
Residualized combined BA        : 0.2273
Residualized process BA         : 0.2101


In [ ]:
# Country-wise audit of residual design-signature leakage
# No neural-network training.

validation_countries = (
    primary_meta.iloc[
        validation_idx
    ]["CNT"]
    .to_numpy()
)


def fit_signature_probe(
    X_train,
    y_train
):

    clf = make_pipeline(

        StandardScaler(),

        LogisticRegression(
            max_iter=500,
            solver="lbfgs",
            class_weight="balanced",
            random_state=GLOBAL_SEED
        )
    )

    clf.fit(
        X_train,
        y_train
    )

    return clf


# Two probes:
# 1. process residual only
# 2. all residual views

probe_process = fit_signature_probe(
    train_residual[
        "process_residual"
    ],
    y_train_signature
)


probe_all = fit_signature_probe(
    X_train_residual_all,
    y_train_signature
)


pred_process = probe_process.predict(
    validation_residual[
        "process_residual"
    ]
)


pred_all = probe_all.predict(
    X_validation_residual_all
)


# Country-level evaluation

country_rows = []


for cnt in sorted(
    np.unique(
        validation_countries
    )
):

    mask = (
        validation_countries
        == cnt
    )


    y_cnt = (
        y_validation_signature[
            mask
        ]
    )


    n_signatures = len(
        np.unique(
            y_cnt
        )
    )


    process_ba = (
        balanced_accuracy_score(
            y_cnt,
            pred_process[
                mask
            ]
        )
    )


    all_ba = (
        balanced_accuracy_score(
            y_cnt,
            pred_all[
                mask
            ]
        )
    )


    country_rows.append(
        {
            "country":
                cnt,

            "n_students":
                int(
                    mask.sum()
                ),

            "n_signatures":
                n_signatures,

            "chance_BA":
                1.0
                / n_signatures,

            "process_residual_BA":
                process_ba,

            "all_residual_views_BA":
                all_ba
        }
    )


country_residual_audit_df = pd.DataFrame(
    country_rows
)


print(
    country_residual_audit_df.to_string(
        index=False
    )
)


print(
    "\nCountry-equal mean BA:"
)


print(
    "Process residual:",
    round(
        country_residual_audit_df[
            "process_residual_BA"
        ].mean(),
        4
    )
)


print(
    "All residual views:",
    round(
        country_residual_audit_df[
            "all_residual_views_BA"
        ].mean(),
        4
    )
)


print(
    "\nPooled validation BA:"
)

print(
    "Process residual: 0.2101"
)

print(
    "All residual views: 0.2273"
)

print(
    "Chance: 0.1000"
)

country  n_students  n_signatures  chance_BA  process_residual_BA  all_residual_views_BA
    ALB        1600            10        0.1             0.206905               0.205178
    ARE        7517            10        0.1             0.213397               0.220277
    FRA        1897            10        0.1             0.185696               0.216329
    ISL         893            10        0.1             0.183210               0.193445
    JOR        2167            10        0.1             0.238078               0.230132
    MEX        1757            10        0.1             0.200346               0.233752
    MKD        1828            10        0.1             0.221666               0.236146
    PAN        1257            10        0.1             0.194917               0.264831
    UZB        2033            10        0.1             0.215162               0.259663

Country-equal mean BA:
Process residual: 0.2066
All residual views: 0.2289

Pooled validation BA:
Process res

In [ ]:
# Country-equal, train-only assessment-design calibration
# No neural-network training.

# Assemble residual representation

X_train_residual = (
    X_train_residual_all.astype(
        np.float32
    )
)

X_validation_residual = (
    X_validation_residual_all.astype(
        np.float32
    )
)


train_signatures = (
    design_signature_ids[
        train_idx
    ]
)

validation_signatures = (
    design_signature_ids[
        validation_idx
    ]
)


train_countries = (
    primary_meta.iloc[
        train_idx
    ]["CNT"]
    .to_numpy()
)


N_SIGNATURES = 10
N_RESIDUAL_FEATURES = (
    X_train_residual.shape[1]
)


# Learn country-equal signature profiles
#
# For every signature:
# 1. compute mean separately in each training country
# 2. average country means equally
#
# This prevents large PISA systems from dominating the
# assessment-design calibration.

signature_profiles = np.zeros(
    (
        N_SIGNATURES,
        N_RESIDUAL_FEATURES
    ),
    dtype=np.float32
)


signature_country_counts = np.zeros(
    N_SIGNATURES,
    dtype=int
)


for signature_id in range(
    N_SIGNATURES
):

    country_means = []


    for country in np.unique(
        train_countries
    ):

        mask = (
            (train_signatures == signature_id)
            &
            (train_countries == country)
        )


        if mask.any():

            country_means.append(
                X_train_residual[
                    mask
                ].mean(
                    axis=0
                )
            )


    country_means = np.vstack(
        country_means
    )


    signature_profiles[
        signature_id
    ] = (
        country_means.mean(
            axis=0
        )
    )


    signature_country_counts[
        signature_id
    ] = (
        len(
            country_means
        )
    )


# Apply fixed training-derived calibration

X_train_design_calibrated = (
    X_train_residual
    -
    signature_profiles[
        train_signatures
    ]
)


X_validation_design_calibrated = (
    X_validation_residual
    -
    signature_profiles[
        validation_signatures
    ]
)


# Audit

print(
    "Training countries contributing "
    "to each signature:"
)

print(
    signature_country_counts
)


print(
    "\nCalibrated train shape:",
    X_train_design_calibrated.shape
)

print(
    "Calibrated validation shape:",
    X_validation_design_calibrated.shape
)


assert (
    signature_country_counts.min()
    == 45
)


print(
    "\nAll 10 design signatures represented "
    "in all 45 training systems: PASS"
)

Training countries contributing to each signature:
[45 45 45 45 45 45 45 45 45 45]

Calibrated train shape: (99966, 21)
Calibrated validation shape: (20949, 21)

All 10 design signatures represented in all 45 training systems: PASS


In [ ]:
# Frozen design probe after country-equal signature calibration

calibrated_result = residual_probe(

    X_train_design_calibrated,

    X_validation_design_calibrated,

    "country_equal_design_calibrated"
)


comparison_df = pd.DataFrame(
    [
        {
            "representation":
                "raw_mask_free",

            "balanced_accuracy":
                0.318307
        },

        {
            "representation":
                "item_residualized",

            "balanced_accuracy":
                0.227259
        },

        {
            "representation":
                "country_equal_design_calibrated",

            "balanced_accuracy":
                calibrated_result[
                    "balanced_accuracy"
                ]
        }
    ]
)


print(
    "Design-signature probe after calibration:"
)

print(
    pd.DataFrame(
        [calibrated_result]
    ).to_string(
        index=False
    )
)


print(
    "\nProgressive design-leakage reduction:"
)

print(
    comparison_df.to_string(
        index=False
    )
)


print(
    "\nChance balanced accuracy: 0.1000"
)

Design-signature probe after calibration:
                    feature_set  n_features  accuracy  balanced_accuracy  macro_f1
country_equal_design_calibrated          21  0.172466           0.172414  0.160257

Progressive design-leakage reduction:
                 representation  balanced_accuracy
                  raw_mask_free           0.318307
              item_residualized           0.227259
country_equal_design_calibrated           0.172414

Chance balanced accuracy: 0.1000


In [ ]:
# Build two disjoint, reasonably balanced CT item sets
# and helper functions for signal-retention analysis.

from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import spearmanr


domain_ids_np = np.asarray(
    item_domain_ids,
    dtype=int
)

process_ids_np = np.asarray(
    item_ideation_ids,
    dtype=int
)

all_items = np.arange(
    N_ITEMS
)


# Find a deterministic 16/16 split that is balanced
# across ideation processes and reasonably balanced
# across domains.

selected_split = None


for seed in range(
    GLOBAL_SEED,
    GLOBAL_SEED + 500
):

    item_A, item_B = train_test_split(

        all_items,

        test_size=0.50,

        random_state=seed,

        stratify=process_ids_np
    )


    domain_A = np.bincount(
        domain_ids_np[item_A],
        minlength=4
    )

    domain_B = np.bincount(
        domain_ids_np[item_B],
        minlength=4
    )


    process_A = np.bincount(
        process_ids_np[item_A],
        minlength=3
    )

    process_B = np.bincount(
        process_ids_np[item_B],
        minlength=3
    )


    if (
        np.all(domain_A >= 2)
        and
        np.all(domain_B >= 2)
        and
        np.max(
            np.abs(
                process_A
                -
                process_B
            )
        ) <= 1
    ):

        selected_split = (
            np.sort(item_A),
            np.sort(item_B),
            seed
        )

        break


assert (
    selected_split is not None
), "Could not construct a suitable item split."


item_A, item_B, split_seed = (
    selected_split
)


print(
    "Item split seed:",
    split_seed
)

print(
    "\nSet A:",
    len(item_A),
    "items"
)

print(
    "Domain counts:",
    np.bincount(
        domain_ids_np[item_A],
        minlength=4
    )
)

print(
    "Process counts:",
    np.bincount(
        process_ids_np[item_A],
        minlength=3
    )
)


print(
    "\nSet B:",
    len(item_B),
    "items"
)

print(
    "Domain counts:",
    np.bincount(
        domain_ids_np[item_B],
        minlength=4
    )
)

print(
    "Process counts:",
    np.bincount(
        process_ids_np[item_B],
        minlength=3
    )
)


# Residual summaries restricted to a specified item set

def build_subset_residual_features(
    indices,
    allowed_items
):

    allowed_items = np.asarray(
        allowed_items
    )

    n = len(indices)

    credit_features = []
    behaviour_features = []
    process_features = []


    for process_id in range(
        N_PROCESSES
    ):

        selected_items = allowed_items[
            process_ids_np[
                allowed_items
            ]
            == process_id
        ]


        assert (
            len(selected_items) > 0
        )


        # CREDIT

        credit_values = []
        credit_masks = []


        for item in selected_items:

            tokens = np.asarray(
                credit_tokens[
                    indices,
                    item
                ]
            )

            mask = np.asarray(
                performance_mask[
                    indices,
                    item
                ],
                dtype=bool
            )


            score = (
                tokens.astype(
                    np.float32
                )
                - 1.0
            ) / 2.0


            residual = (
                score
                -
                credit_item_mean[item]
            )


            residual[
                ~mask
            ] = 0.0


            credit_values.append(
                residual
            )

            credit_masks.append(
                mask
            )


        credit_values = np.column_stack(
            credit_values
        )

        credit_masks = np.column_stack(
            credit_masks
        )


        denominator = (
            credit_masks.sum(
                axis=1
            )
        )


        credit_mean = np.divide(

            (
                credit_values
                * credit_masks
            ).sum(
                axis=1
            ),

            denominator,

            out=np.zeros(
                n,
                dtype=np.float32
            ),

            where=(
                denominator > 0
            )
        )


        credit_features.append(
            credit_mean
        )


        # BEHAVIOUR

        for behaviour_k, token in enumerate(
            [1, 2, 3]
        ):

            values_list = []
            masks_list = []


            for item in selected_items:

                b = np.asarray(
                    behaviour_tokens[
                        indices,
                        item
                    ]
                )

                mask = np.asarray(
                    interaction_mask[
                        indices,
                        item
                    ],
                    dtype=bool
                )


                residual = (

                    (
                        b == token
                    ).astype(
                        np.float32
                    )

                    -

                    behaviour_item_rates[
                        item,
                        behaviour_k
                    ]
                )


                residual[
                    ~mask
                ] = 0.0


                values_list.append(
                    residual
                )

                masks_list.append(
                    mask
                )


            values_matrix = np.column_stack(
                values_list
            )

            masks_matrix = np.column_stack(
                masks_list
            )


            denominator = (
                masks_matrix.sum(
                    axis=1
                )
            )


            mean_residual = np.divide(

                (
                    values_matrix
                    * masks_matrix
                ).sum(
                    axis=1
                ),

                denominator,

                out=np.zeros(
                    n,
                    dtype=np.float32
                ),

                where=(
                    denominator > 0
                )
            )


            behaviour_features.append(
                mean_residual
            )


        # PROCESS

        for channel in range(
            N_PROCESS_CHANNELS
        ):

            values_list = []
            masks_list = []


            for item in selected_items:

                x = np.asarray(
                    process_tensor[
                        indices,
                        item,
                        channel
                    ],
                    dtype=np.float32
                )


                mask = np.isfinite(
                    x
                )


                residual = np.zeros(
                    n,
                    dtype=np.float32
                )


                residual[
                    mask
                ] = (

                    x[
                        mask
                    ]

                    -

                    process_item_mean[
                        item,
                        channel
                    ]

                ) / (

                    process_item_std[
                        item,
                        channel
                    ]

                    + EPS
                )


                values_list.append(
                    residual
                )

                masks_list.append(
                    mask
                )


            values_matrix = np.column_stack(
                values_list
            )

            masks_matrix = np.column_stack(
                masks_list
            )


            denominator = (
                masks_matrix.sum(
                    axis=1
                )
            )


            mean_residual = np.divide(

                (
                    values_matrix
                    * masks_matrix
                ).sum(
                    axis=1
                ),

                denominator,

                out=np.zeros(
                    n,
                    dtype=np.float32
                ),

                where=(
                    denominator > 0
                )
            )


            process_features.append(
                mean_residual
            )


    return np.column_stack(
        [
            np.column_stack(
                credit_features
            ),

            np.column_stack(
                behaviour_features
            ),

            np.column_stack(
                process_features
            )
        ]
    ).astype(
        np.float32
    )


print(
    "\nDisjoint-item feature builder ready."
)

Item split seed: 20261005

Set A: 16 items
Domain counts: [8 2 3 3]
Process counts: [6 6 4]

Set B: 16 items
Domain counts: [4 2 7 3]
Process counts: [6 5 5]

Disjoint-item feature builder ready.


In [ ]:
# Test whether design calibration preserves
# creative-performance signal on disjoint held-out items.

def heldout_credit_target(
    indices,
    target_items,
    min_scorable_items=3
):

    tokens = np.asarray(
        credit_tokens[
            indices
        ][:, target_items]
    )


    mask = np.asarray(
        performance_mask[
            indices
        ][:, target_items],
        dtype=bool
    )


    scores = (
        tokens.astype(
            np.float32
        )
        - 1.0
    ) / 2.0


    n_scorable = (
        mask.sum(
            axis=1
        )
    )


    target = np.divide(

        (
            scores
            * mask
        ).sum(
            axis=1
        ),

        n_scorable,

        out=np.full(
            len(indices),
            np.nan,
            dtype=np.float32
        ),

        where=(
            n_scorable > 0
        )
    )


    eligible = (
        n_scorable
        >= min_scorable_items
    )


    return (
        target,
        eligible,
        n_scorable
    )


def learn_signature_profiles(
    X_train,
    signatures,
    countries
):

    profiles = np.zeros(
        (
            10,
            X_train.shape[1]
        ),
        dtype=np.float32
    )


    for signature_id in range(
        10
    ):

        country_means = []


        for country in np.unique(
            countries
        ):

            mask = (
                (signatures == signature_id)
                &
                (countries == country)
            )


            if mask.any():

                country_means.append(
                    X_train[
                        mask
                    ].mean(
                        axis=0
                    )
                )


        profiles[
            signature_id
        ] = np.vstack(
            country_means
        ).mean(
            axis=0
        )


    return profiles


def run_signal_retention_direction(
    source_items,
    target_items,
    direction_name
):

    # Source representations

    X_train = (
        build_subset_residual_features(
            train_idx,
            source_items
        )
    )


    X_val = (
        build_subset_residual_features(
            validation_idx,
            source_items
        )
    )


    # Design calibration learned on training systems only

    profiles = learn_signature_profiles(

        X_train,

        train_signatures,

        train_countries
    )


    X_train_cal = (
        X_train
        -
        profiles[
            train_signatures
        ]
    )


    X_val_cal = (
        X_val
        -
        profiles[
            validation_signatures
        ]
    )


    # Held-out target items

    y_train, eligible_train, _ = (
        heldout_credit_target(
            train_idx,
            target_items
        )
    )


    y_val, eligible_val, _ = (
        heldout_credit_target(
            validation_idx,
            target_items
        )
    )


    # Predictive probe

    rows = []


    for representation_name, Xt, Xv in [

        (
            "item_residualized",
            X_train,
            X_val
        ),

        (
            "design_calibrated",
            X_train_cal,
            X_val_cal
        )
    ]:


        model_probe = make_pipeline(

            StandardScaler(),

            RidgeCV(
                alphas=[
                    0.1,
                    1.0,
                    10.0,
                    100.0
                ]
            )
        )


        model_probe.fit(

            Xt[
                eligible_train
            ],

            y_train[
                eligible_train
            ]
        )


        pred = model_probe.predict(

            Xv[
                eligible_val
            ]
        )


        y_true = (
            y_val[
                eligible_val
            ]
        )


        rho = spearmanr(
            y_true,
            pred
        ).statistic


        rows.append(
            {
                "direction":
                    direction_name,

                "representation":
                    representation_name,

                "n_train":
                    int(
                        eligible_train.sum()
                    ),

                "n_validation":
                    int(
                        eligible_val.sum()
                    ),

                "R2":
                    r2_score(
                        y_true,
                        pred
                    ),

                "MAE":
                    mean_absolute_error(
                        y_true,
                        pred
                    ),

                "Spearman_rho":
                    rho
            }
        )


    return rows


# A -> B and B -> A

signal_rows = []


signal_rows.extend(

    run_signal_retention_direction(

        source_items=item_A,

        target_items=item_B,

        direction_name="A_to_B"
    )
)


signal_rows.extend(

    run_signal_retention_direction(

        source_items=item_B,

        target_items=item_A,

        direction_name="B_to_A"
    )
)


signal_retention_df = pd.DataFrame(
    signal_rows
)


print(
    signal_retention_df.to_string(
        index=False
    )
)


# Mean across the two disjoint directions

signal_summary_df = (

    signal_retention_df

    .groupby(
        "representation"
    )[
        [
            "R2",
            "MAE",
            "Spearman_rho"
        ]
    ]

    .mean()

    .reset_index()
)


print(
    "\nMean across A→B and B→A:"
)

print(
    signal_summary_df.to_string(
        index=False
    )
)

direction    representation  n_train  n_validation       R2      MAE  Spearman_rho
   A_to_B item_residualized    94046         19312 0.394539 0.171185      0.646792
   A_to_B design_calibrated    94046         19312 0.393885 0.171419      0.646920
   B_to_A item_residualized    95332         19370 0.421657 0.160401      0.662274
   B_to_A design_calibrated    95332         19370 0.422211 0.160433      0.662967

Mean across A→B and B→A:
   representation       R2      MAE  Spearman_rho
design_calibrated 0.408048 0.165926      0.654943
item_residualized 0.408098 0.165793      0.654533


In [ ]:
# Save final design-calibration artefacts and safeguard results

REFINEMENT_DIR = (
    BASE_DIR
    / "processed"
    / "crest_design_calibration"
)

REFINEMENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Save fold-0 calibration parameters

np.save(
    REFINEMENT_DIR
    / "fold0_credit_item_mean.npy",
    credit_item_mean
)

np.save(
    REFINEMENT_DIR
    / "fold0_behaviour_item_rates.npy",
    behaviour_item_rates
)

np.save(
    REFINEMENT_DIR
    / "fold0_process_item_mean.npy",
    process_item_mean
)

np.save(
    REFINEMENT_DIR
    / "fold0_process_item_std.npy",
    process_item_std
)

np.save(
    REFINEMENT_DIR
    / "fold0_signature_profiles.npy",
    signature_profiles
)

np.save(
    REFINEMENT_DIR
    / "booklet_to_signature.npy",
    BOOKLET_TO_SIGNATURE
)


# Save diagnostic results

comparison_df.to_csv(
    RESULTS_DIR
    / "fold0_design_leakage_progression.csv",
    index=False
)

country_residual_audit_df.to_csv(
    RESULTS_DIR
    / "fold0_countrywise_residual_design_leakage.csv",
    index=False
)

signal_retention_df.to_csv(
    RESULTS_DIR
    / "fold0_design_calibration_signal_retention.csv",
    index=False
)

signal_summary_df.to_csv(
    RESULTS_DIR
    / "fold0_design_calibration_signal_retention_summary.csv",
    index=False
)


# Concise summary

summary = {
    "raw_mask_free_design_BA": 0.318307,
    "item_residualized_design_BA": 0.227259,
    "design_calibrated_design_BA": 0.172414,

    "heldout_R2_before": float(
        signal_summary_df.loc[
            signal_summary_df[
                "representation"
            ] == "item_residualized",
            "R2"
        ].iloc[0]
    ),

    "heldout_R2_after": float(
        signal_summary_df.loc[
            signal_summary_df[
                "representation"
            ] == "design_calibrated",
            "R2"
        ].iloc[0]
    ),

    "heldout_Spearman_before": float(
        signal_summary_df.loc[
            signal_summary_df[
                "representation"
            ] == "item_residualized",
            "Spearman_rho"
        ].iloc[0]
    ),

    "heldout_Spearman_after": float(
        signal_summary_df.loc[
            signal_summary_df[
                "representation"
            ] == "design_calibrated",
            "Spearman_rho"
        ].iloc[0]
    )
}


with open(
    REFINEMENT_DIR
    / "fold0_calibration_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print(
    "Saved calibration artefacts to:"
)

print(
    REFINEMENT_DIR
)

print(
    "\nFinal diagnostic summary:"
)

for key, value in summary.items():

    print(
        f"{key:38s}",
        f"{value:.6f}"
    )

Saved calibration artefacts to:
PROJECT_ROOT\processed\crest_design_calibration

Final diagnostic summary:
raw_mask_free_design_BA                0.318307
item_residualized_design_BA            0.227259
design_calibrated_design_BA            0.172414
heldout_R2_before                      0.408098
heldout_R2_after                       0.408048
heldout_Spearman_before                0.654533
heldout_Spearman_after                 0.654943
